# Exercício 17 — notícias do dia, Pydantic e resumo executivo (sem ecrã)

**Objectivos pedagógicos**

- **Tool de pesquisa:** `pesquisa_noticias_web` (DuckDuckGo via `langchain_community`) para ir buscar trechos da Web — **rede obrigatória**.
- **Agente 1 (pesquisa):** `create_agent` + Gemini com instruções de jornalista; decide quantas consultas fazer.
- **Dados estruturados:** `BoletimNoticias` / `NoticiaItem` (Pydantic v2) via `with_structured_output`.
- **Indicadores:** contagens, média de relevância, proporção internacional, diversidade de categorias (cálculo determinístico em Python).
- **Agente 2 (redacção):** transforma o JSON do boletim em **Markdown** executivo (sem *tools*).
- **Resumo executivo estruturado:** modelo `ResumoExecutivo` (Pydantic) alinhado com o boletim.

**Chaves:** `GOOGLE_API_KEY` no `.env` na raiz; opcional `GEMINI_MODEL_EX17` (senão `GEMINI_MODEL`). Arranque: `./run.sh` nesta pasta.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
REPO = ROOT.parent.parent
load_dotenv(REPO / ".env", override=False)

_ex = ROOT.parent
if str(_ex) not in sys.path:
    sys.path.insert(0, str(_ex))

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no `.env` na raiz do repositório.")

from noticias_agentes import executar_pipeline_noticias

CONSULTA = os.environ.get("EX17_CONSULTA", "notícias de hoje Portugal economia política sociedade")
print("Consulta:", CONSULTA, flush=True)

/opt/conda/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Consulta: notícias de hoje Portugal economia política sociedade


In [2]:
resultado = executar_pipeline_noticias(CONSULTA)

print("\n=== Boletim (Pydantic → JSON) ===\n")
print(resultado["boletim"].model_dump_json(indent=2, ensure_ascii=False))

print("\n=== Indicadores ===\n")
ind = resultado["indicadores"]
display(pd.DataFrame([ind.model_dump()]))

df_cat = (
    pd.DataFrame(
        [(k, v) for k, v in ind.contagem_por_categoria.items()],
        columns=["categoria", "n_noticias"],
    )
    if ind.contagem_por_categoria
    else pd.DataFrame(columns=["categoria", "n_noticias"])
)
if not df_cat.empty:
    print("Distribuição por categoria:")
    display(df_cat)


=== Boletim (Pydantic → JSON) ===

{
  "data_referencia": "2026-05-09",
  "consulta_original": "notícias de hoje Portugal economia política sociedade",
  "itens": [
    {
      "titulo": "Aumento do salário mínimo nacional para 820 euros em 2024",
      "url": "https://eco.sapo.pt",
      "fonte": "eco.sapo.pt",
      "resumo_curto": "Aumento do salário mínimo nacional para 820 euros em 2024, com impacto nas contas das empresas",
      "categoria": "economia",
      "relevancia_0_10": 7
    },
    {
      "titulo": "Marcelo Rebelo de Sousa defende estabilidade política",
      "url": "https://www.jornaldenegocios.pt/",
      "fonte": "jornaldenegocios.pt",
      "resumo_curto": "Marcelo Rebelo de Sousa defende estabilidade política para o país, após aprovação do Orçamento do Estado",
      "categoria": "outra",
      "relevancia_0_10": 6
    },
    {
      "titulo": "Greve de professores causa protestos",
      "url": "https://www.publico.pt/",
      "fonte": "publico.pt",
      "resu

,total_noticias,contagem_por_categoria,relevancia_media,fraccao_internacional,diversidade_categorias,titulo_maior_relevancia
0,4,"{'economia': 1, 'outra': 2, 'sociedade': 1}",5.5,0.0,3,Aumento do salário mínimo nacional para 820 eu...


Distribuição por categoria:


,categoria,n_noticias
0,economia,1
1,outra,2
2,sociedade,1


In [3]:
print("=== Resumo executivo (Pydantic) ===\n")
print(resultado["resumo_executivo_estruturado"].model_dump_json(indent=2, ensure_ascii=False))

print("\n=== Resumo executivo (Markdown — agente redactor) ===\n")
print(resultado["resumo_executivo_markdown"])

=== Resumo executivo (Pydantic) ===

{
  "headline": "Resumo Executivo - Notícias de Portugal (09/05/2026)",
  "pontos_chave": [
    "Aumento do salário mínimo nacional para 820 euros em 2024, impactando as empresas.",
    "O Presidente Marcelo Rebelo de Sousa enfatiza a importância da estabilidade política.",
    "Greve de professores gera protestos e negociações com o governo.",
    "Discussão em torno de novas medidas de apoio à habitação."
  ],
  "riscos_ou_incertezas": [
    "Possível impacto negativo do aumento do salário mínimo nas contas das empresas.",
    "Instabilidade política pode afetar a implementação de políticas governamentais.",
    "Continuação da greve dos professores pode prejudicar o sistema educacional.",
    "Divergências políticas podem dificultar a aprovação de medidas de apoio à habitação."
  ],
  "oportunidades_ou_acoes": [],
  "disclaimer": "Confirmar nas fontes; resumo automático pedagógico."
}

=== Resumo executivo (Markdown — agente redactor) ===

## Res

## Notas

- A qualidade depende dos **snippets** devolvidos pelo DuckDuckGo; em produção costuma usar-se um motor de notícias com API (ex. Tavily, NewsAPI).
- O relatório bruto do agente de pesquisa está em `resultado["relatorio_pesquisa_bruto"]` para depuração.